# U21 — ETL & Orchestration: Lab

### Real-world brief: an ETL pipeline for a wind farm's SCADA & maintenance data

A wind-farm operator has three **source systems**, all imperfect: hourly **SCADA telemetry** (with gaps, nulls, duplicates and bad sensor values), an **asset registry**, and a **maintenance log** (in Excel). The analytics team needs a clean **daily performance table** in a warehouse — and it must be safe to re-run, load only new data, and fail loudly when the data is wrong.

You'll build the full pipeline: **Extract → Transform → Quality-check → Load**, make it **idempotent** and **incremental**, and wrap it in a tiny **orchestration DAG** with retries — the real day-to-day of data engineering.

**Resources provided:** `turbine_telemetry.csv`, `turbine_registry.csv`, `maintenance_log.xlsx`. We load into a local **SQLite** file as the 'warehouse' (no server needed).

_Phase F — Data Engineering._

#objectives

Extract from heterogeneous sources (CSV + Excel)

Transform: clean, validate, join and aggregate to a fact table

Load to a warehouse with idempotent UPSERTs

Run incremental loads using a high-water mark

Add data-quality checks and orchestrate tasks as a DAG with retries

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [ ]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_windfarm(tele_path="turbine_telemetry.csv", reg_path="turbine_registry.csv",
                   maint_path="maintenance_log.xlsx", seed=211, verbose=False):
    """Raw operational data for a wind farm — the messy SOURCE systems an ETL pipeline must
    ingest (U21). Three sources, deliberately imperfect like real SCADA exports:

      turbine_telemetry.csv  hourly SCADA readings (with gaps, nulls, bad values, dupes)
      turbine_registry.csv   asset master (one row per turbine)
      maintenance_log.xlsx   work orders / downtime events
    """
    rng = np.random.default_rng(seed)

    # ---- asset registry (8 turbines across 2 sites) ----
    turbines = [f"T{i:02d}" for i in range(1, 9)]
    sites = ["Kutch", "Kutch", "Kutch", "Kutch", "Satara", "Satara", "Satara", "Satara"]
    models = rng.choice(["GE-2.0", "Vestas-2.5", "Suzlon-2.1"], 8)
    rated = np.array([2000, 2000, 2500, 2500, 2100, 2100, 2000, 2500])
    reg = pd.DataFrame({
        "turbine_id": turbines, "site": sites, "model": models,
        "rated_power_kw": rated,
        "commission_date": pd.to_datetime("2019-01-01") + pd.to_timedelta(rng.integers(0, 1500, 8), unit="D"),
        "latitude": np.round(rng.uniform(17, 23, 8), 4),
        "longitude": np.round(rng.uniform(69, 74, 8), 4),
    })
    reg.to_csv(reg_path, index=False)

    # ---- hourly telemetry over 30 days ----
    dates = pd.date_range("2024-05-01", "2024-05-30 23:00", freq="h")
    rows = []
    for ti, tid in enumerate(turbines):
        rp = rated[ti]
        wind = np.clip(rng.weibull(2.0, len(dates)) * 6.5, 0, 25)
        # power curve: cut-in 3, rated ~12 m/s, cut-out 25
        pc = np.clip((wind - 3) / (12 - 3), 0, 1) ** 3
        pc[wind < 3] = 0; pc[wind > 25] = 0
        power = pc * rp * rng.normal(1.0, 0.05, len(dates))
        power = np.clip(power, 0, rp * 1.05)
        rpm = np.clip(wind * 1.3 + rng.normal(0, 0.5, len(dates)), 0, 18)
        amb = 28 + 6 * np.sin(np.arange(len(dates)) / 24 * 2 * np.pi) + rng.normal(0, 1.5, len(dates))
        gearbox = amb + 0.018 * power + rng.normal(0, 2, len(dates))
        nacelle = amb + 0.010 * power + rng.normal(0, 1.5, len(dates))
        df = pd.DataFrame({
            "timestamp": dates, "turbine_id": tid,
            "wind_speed_ms": wind.round(2), "power_kw": power.round(1),
            "rotor_rpm": rpm.round(2), "gearbox_temp_c": gearbox.round(1),
            "nacelle_temp_c": nacelle.round(1),
        })
        rows.append(df)
    tele = pd.concat(rows, ignore_index=True)

    # ---- inject realistic messiness ----
    n = len(tele)
    # 1) missing rows (sensor dropout) -> availability gaps
    drop = rng.random(n) < 0.03
    tele = tele[~drop].reset_index(drop=True); n = len(tele)
    # 2) null sensor values
    for col in ["gearbox_temp_c", "rotor_rpm", "wind_speed_ms"]:
        tele.loc[rng.random(n) < 0.01, col] = np.nan
    # 3) impossible values
    bad = rng.random(n) < 0.004
    tele.loc[bad, "power_kw"] = rng.choice([-50, -999, 99999], bad.sum())
    # 4) duplicate rows
    dups = tele.sample(60, random_state=seed)
    tele = pd.concat([tele, dups], ignore_index=True)
    # 5) shuffle (timestamps arrive unsorted)
    tele = tele.sample(frac=1, random_state=seed).reset_index(drop=True)
    tele.to_csv(tele_path, index=False)

    # ---- maintenance work orders ----
    nwo = 120
    wo = pd.DataFrame({
        "work_order_id": [f"WO-{1000+i}" for i in range(nwo)],
        "turbine_id": rng.choice(turbines, nwo),
        "start_date": pd.to_datetime("2024-05-01") + pd.to_timedelta(rng.integers(0, 30, nwo), unit="D"),
        "type": rng.choice(["scheduled", "corrective"], nwo, p=[0.45, 0.55]),
        "downtime_hours": np.round(rng.gamma(2.0, 3.0, nwo), 1),
        "cost_inr": (rng.gamma(2.0, 25000, nwo)).round(0).astype(int),
    })
    wo["end_date"] = wo["start_date"] + pd.to_timedelta(wo["downtime_hours"], unit="h")
    wo = wo[["work_order_id", "turbine_id", "start_date", "end_date", "type", "downtime_hours", "cost_inr"]]
    wo.to_excel(maint_path, index=False, sheet_name="work_orders")

    if verbose:
        print("telemetry:", tele.shape, "| nulls:", int(tele.isna().sum().sum()),
              "| dupes:", int(tele.duplicated().sum()),
              "| bad power rows:", int(((tele.power_kw < 0) | (tele.power_kw > 5000)).sum()))
        print("registry:", reg.shape, "| maintenance:", wo.shape)
        print("date span:", tele.timestamp.min(), "->", tele.timestamp.max())
    return tele, reg, wo

if not (os.path.exists('turbine_telemetry.csv') and os.path.exists('turbine_registry.csv')
        and os.path.exists('maintenance_log.xlsx')):
    build_windfarm(); print('Generated source files.')
else:
    print('Found the provided source files.')

In [ ]:
import pandas as pd, numpy as np, sqlite3
pd.set_option('display.width', 120)
WAREHOUSE = 'windfarm_warehouse.db'   # our local 'data warehouse'

#1. Extract — read the raw sources

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. EXTRACT: pull each source and inspect it
# -----------------------------------------------------------
def extract():
    tele = pd.read_csv('turbine_telemetry.csv', parse_dates=['timestamp'])
    reg = pd.read_csv('turbine_registry.csv', parse_dates=['commission_date'])
    maint = pd.read_excel('maintenance_log.xlsx', parse_dates=['start_date', 'end_date'])
    return tele, reg, maint

tele, reg, maint = extract()
print('telemetry:', tele.shape, '| registry:', reg.shape, '| maintenance:', maint.shape)
print('\ntelemetry dtypes:'); print(tele.dtypes)
tele.head(3)

In [ ]:
# A quick data-health scan reveals the mess we must clean
print('null values per column:'); print(tele.isna().sum())
print('\nduplicate rows:', int(tele.duplicated().sum()))
print('power_kw range:', tele.power_kw.min(), 'to', tele.power_kw.max(), '(negatives & spikes = bad)')

#2. Transform — clean, join, aggregate

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. CLEAN the telemetry
# -----------------------------------------------------------
def clean_telemetry(tele, reg):
    df = tele.drop_duplicates().copy()                       # 1) drop exact dupes
    df = df.merge(reg[['turbine_id', 'rated_power_kw', 'site']], on='turbine_id', how='left')
    # 2) remove impossible power values (negative or above rated headroom)
    df = df[(df.power_kw >= 0) & (df.power_kw <= df.rated_power_kw * 1.1)]
    # 3) drop rows missing the fields we need downstream
    df = df.dropna(subset=['power_kw', 'wind_speed_ms'])
    # 4) derive fields
    df['date'] = df['timestamp'].dt.date
    df['capacity_factor'] = df['power_kw'] / df['rated_power_kw']
    return df

clean = clean_telemetry(tele, reg)
print('rows after cleaning:', len(clean), '(from', len(tele), ')')
print('power_kw range now:', round(clean.power_kw.min(), 1), 'to', round(clean.power_kw.max(), 1))

In [ ]:
# -----------------------------------------------------------
# 🔹 2B. AGGREGATE to a daily fact table per turbine
# -----------------------------------------------------------
EXPECTED_PER_DAY = 24   # hourly readings -> 24 expected intervals/day
def build_daily_fact(clean, maint):
    g = clean.groupby(['turbine_id', 'date'])
    fact = g.agg(avg_power_kw=('power_kw', 'mean'),
                 energy_kwh=('power_kw', 'sum'),          # hourly kW summed = kWh
                 avg_wind_ms=('wind_speed_ms', 'mean'),
                 max_gearbox_c=('gearbox_temp_c', 'max'),
                 capacity_factor=('capacity_factor', 'mean'),
                 intervals=('power_kw', 'count')).reset_index()
    fact['availability_pct'] = (fact['intervals'] / EXPECTED_PER_DAY * 100).clip(upper=100).round(1)
    # join maintenance downtime per turbine per day
    m = maint.copy(); m['date'] = m['start_date'].dt.date
    dt = m.groupby(['turbine_id', 'date'])['downtime_hours'].sum().reset_index()
    fact = fact.merge(dt, on=['turbine_id', 'date'], how='left')
    fact['downtime_hours'] = fact['downtime_hours'].fillna(0)
    for c in ['avg_power_kw', 'energy_kwh', 'avg_wind_ms', 'capacity_factor']:
        fact[c] = fact[c].round(3)
    return fact

fact = build_daily_fact(clean, maint)
print('daily fact rows:', fact.shape)
fact.head()

#### 🧪 EXERCISE 1 — Add a transform
Extend `build_daily_fact` (or post-process `fact`) to add a **`load_factor_flag`** column: `'underperforming'` when `capacity_factor < 0.25`, else `'normal'`.
1. Add the column and print the count of underperforming turbine-days.
2. In a comment, explain why deriving such flags in the transform step (not at query time) keeps downstream dashboards simple and consistent.

In [ ]:
# 1. add load_factor_flag and count underperforming days
# YOUR CODE HERE

# 2. why derive in transform: ...   (comment)

#3. Load — write to the warehouse, idempotently

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. CREATE the warehouse table with a primary key
# The PK (turbine_id, date) is what makes UPSERT possible.
# -----------------------------------------------------------
def init_warehouse(db=WAREHOUSE):
    con = sqlite3.connect(db)
    con.execute('''CREATE TABLE IF NOT EXISTS daily_performance (
        turbine_id TEXT, date TEXT, avg_power_kw REAL, energy_kwh REAL,
        avg_wind_ms REAL, max_gearbox_c REAL, capacity_factor REAL,
        intervals INTEGER, availability_pct REAL, downtime_hours REAL,
        PRIMARY KEY (turbine_id, date) )''')
    con.commit(); con.close()

init_warehouse()
print('warehouse table ready.')

In [ ]:
# -----------------------------------------------------------
# 🔹 3B. IDEMPOTENT LOAD via UPSERT (INSERT ... ON CONFLICT DO UPDATE)
# Re-running with the same data must NOT create duplicates.
# -----------------------------------------------------------
def upsert_fact(fact, db=WAREHOUSE):
    cols = ['turbine_id','date','avg_power_kw','energy_kwh','avg_wind_ms','max_gearbox_c',
            'capacity_factor','intervals','availability_pct','downtime_hours']
    rows = fact.assign(date=fact['date'].astype(str))[cols].itertuples(index=False, name=None)
    sql = f'''INSERT INTO daily_performance ({','.join(cols)})
              VALUES ({','.join(['?']*len(cols))})
              ON CONFLICT(turbine_id, date) DO UPDATE SET
              {', '.join(f'{c}=excluded.{c}' for c in cols[2:])}'''
    con = sqlite3.connect(db)
    con.executemany(sql, rows); con.commit()
    n = con.execute('SELECT COUNT(*) FROM daily_performance').fetchone()[0]
    con.close(); return n

n1 = upsert_fact(fact)
n2 = upsert_fact(fact)   # run AGAIN — idempotent, so the count must stay the same
print('rows after first load:', n1)
print('rows after second load:', n2, '(unchanged -> idempotent!)')

#### 🧪 EXERCISE 2 — Prove idempotency
1. Pick one turbine-day already in the warehouse, query its `energy_kwh`.
2. Change that value in a copy of `fact`, `upsert_fact` again, and re-query — confirm the row was **updated, not duplicated**, and the total row count is unchanged.
3. In a comment, explain why idempotency lets you safely retry a failed pipeline run.

In [ ]:
# 1-2. query, modify, upsert, re-query; confirm update-not-insert
# YOUR CODE HERE

# 3. why idempotency enables safe retries: ...   (comment)

#4. Incremental loads with a high-water mark

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. Only process dates NEWER than what's already loaded
# -----------------------------------------------------------
def high_water_mark(db=WAREHOUSE):
    con = sqlite3.connect(db)
    hw = con.execute('SELECT MAX(date) FROM daily_performance').fetchone()[0]
    con.close(); return hw

def run_incremental(db=WAREHOUSE):
    tele, reg, maint = extract()
    clean = clean_telemetry(tele, reg)
    fact = build_daily_fact(clean, maint)
    hw = high_water_mark(db)
    if hw is not None:
        before = len(fact)
        fact = fact[fact['date'].astype(str) > hw]   # keep only new partitions
        print(f'high-water mark = {hw}; {before} -> {len(fact)} new rows to load')
    return upsert_fact(fact, db) if len(fact) else None

# Simulate: warehouse already has everything, so an incremental run finds nothing new
print('high-water mark currently:', high_water_mark())
run_incremental()
print('A fresh run loads only new dates — cheap and fast.')

#### 🧪 EXERCISE 3 — Backfill a new day
Simulate new data arriving for a date **after** the current high-water mark.
1. Take a copy of `fact`, shift its `date` forward (e.g. all rows to `2024-06-01`), and run the incremental load logic so only those new-date rows are inserted.
2. Confirm the warehouse row count grew by exactly the number of new turbine-days.
3. In a comment, explain why incremental loads matter once a table has billions of rows.

In [ ]:
# 1-2. craft a new-date partition and load only the new rows
# YOUR CODE HERE

# 3. why incremental loads matter at scale: ...   (comment)

#5. Data-quality checks

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. A tiny quality-check framework — fail loudly on bad data
# -----------------------------------------------------------
class DataQualityError(Exception):
    pass

def run_quality_checks(fact):
    checks = {
        'non_empty': len(fact) > 0,
        'keys_not_null': fact[['turbine_id', 'date']].notna().all().all(),
        'capacity_factor_in_range': fact['capacity_factor'].between(0, 1.1).all(),
        'availability_in_range': fact['availability_pct'].between(0, 100).all(),
        'no_negative_energy': (fact['energy_kwh'] >= 0).all(),
    }
    failed = [name for name, ok in checks.items() if not ok]
    for name, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    if failed:
        raise DataQualityError(f'quality checks failed: {failed}')
    return True

run_quality_checks(fact)
print('all quality checks passed.')

#### 🧪 EXERCISE 4 — Write a freshness check
Stale data is a silent failure. 
1. Add a check `data_is_fresh`: the max `date` in `fact` is within, say, 2 days of the latest `timestamp` in the raw telemetry (compute both and compare).
2. Deliberately break it (filter `fact` to old dates) and confirm `run_quality_checks` raises.
3. In a comment, name two other checks you'd add for a production pipeline.

In [ ]:
# 1-2. add a freshness check and show it failing on stale data
# YOUR CODE HERE

# 3. two more production checks: ...   (comment)

#6. Orchestration — a mini DAG with retries

In [ ]:
# -----------------------------------------------------------
# 🔹 6A. Define tasks + dependencies and run them in order
# This is the essence of Airflow: a DAG of tasks, each retried on failure.
# -----------------------------------------------------------
import time, traceback

DAG = {
    'extract':       [],
    'transform':     ['extract'],
    'quality_check': ['transform'],
    'load':          ['quality_check'],
}

def topological_order(dag):
    order, seen = [], set()
    def visit(t):
        if t in seen: return
        for dep in dag[t]: visit(dep)
        seen.add(t); order.append(t)
    for t in dag: visit(t)
    return order

def run_dag(dag, tasks, max_retries=2):
    ctx = {}
    for t in topological_order(dag):
        for attempt in range(1, max_retries + 2):
            try:
                tasks[t](ctx)
                print(f'  [OK]   {t}')
                break
            except Exception as e:
                print(f'  [RETRY {attempt}] {t} failed: {e}')
                if attempt == max_retries + 1:
                    print(f'  [FAIL] {t} gave up after {attempt} attempts'); raise
                time.sleep(0.1)
    return ctx

print('task order:', topological_order(DAG))

In [ ]:
# Wire the pipeline functions into DAG tasks (they share a context dict)
def task_extract(ctx):  ctx['tele'], ctx['reg'], ctx['maint'] = extract()
def task_transform(ctx): ctx['fact'] = build_daily_fact(clean_telemetry(ctx['tele'], ctx['reg']), ctx['maint'])
def task_quality(ctx):   run_quality_checks(ctx['fact'])
def task_load(ctx):      ctx['rows'] = upsert_fact(ctx['fact'])
TASKS = {'extract': task_extract, 'transform': task_transform,
         'quality_check': task_quality, 'load': task_load}

print('Running the pipeline DAG:')
ctx = run_dag(DAG, TASKS)
print('pipeline complete — warehouse rows:', ctx['rows'])

#### 🧪 EXERCISE 5 — A flaky task that recovers, and a new node
1. Write a task that fails on its first attempt but succeeds on a retry (use a counter / `random`), insert it into the DAG, and watch `run_dag` recover via the retry logic.
2. Add a new leaf task `notify` that depends on `load` and just prints a success summary (rows loaded). Re-run the DAG.
3. In a comment, map each piece of this mini-runner to its Airflow equivalent (DAG, task, dependency, retries).

In [ ]:
# 1-2. flaky-but-recovering task + a 'notify' leaf task; re-run the DAG
import random
# YOUR CODE HERE

# 3. mapping to Airflow concepts: ...   (comment)

#📘 Summary

| Stage | What you built |
| ----- | -------------- |
| Extract | read CSV + Excel sources, scanned data health |
| Transform | cleaned bad/dup/null rows, joined, aggregated to a daily fact |
| Load | idempotent UPSERT into a SQLite warehouse (re-runnable) |
| Incremental | high-water mark loads only new partitions |
| Quality | a check framework that fails loudly on bad data |
| Orchestrate | a DAG of tasks run in order, with retries |

**Core lesson:** a good pipeline is **reliability engineering** — idempotent so you can retry, incremental so it scales, validated so bad data is caught, and orchestrated so it runs itself. Tools like Airflow are this same DAG-of-tasks idea, productionised.

**Next — U22:** when the data outgrows one machine, move to streaming (Kafka) and distributed compute (Spark).